In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import umap

In [ ]:
def load_data(tissue: str):
    proteins = pd.read_csv(f"../../data/raw/{tissue}__proteins.csv")
    samples = pd.read_csv(f"../../data/raw/{tissue}__samples.csv") # key column: nhpi
    quant = pd.read_csv(f"../../data/raw/{tissue}__quant.csv")

    protein_mapping = proteins[["Protein.Group", "Genes"]].drop_duplicates()
    quant = quant.merge(protein_mapping, on="Protein.Group", how="left")
    quant = quant.set_index("Protein.Group").T
    quant = quant.loc[:, ~quant.columns.astype(str).str.contains(";", regex=False)]
    proteins = list(quant.columns)
    df = samples.merge(quant, left_on="Run", right_index=True)
    return df, proteins

tissues = ["heart", "kidney", "leukocytes", "liver", "lungs", "plasma", "spleen"]

all_proteins = set()
dfs = []
for tissue in tissues:
    df, proteins = load_data(tissue)
    dfs.append(df)
    all_proteins.update(proteins)
all_proteins = list(all_proteins)
print(all_proteins[:5])
dfs[1].head()

In [ ]:
# merge datasets
merged_dfs = []
for df in dfs:
    missing_proteins = set(all_proteins) - set(df.columns)
    for p in missing_proteins:
        df[p] = np.nan
    merged_dfs.append(df[["nhpi", "Model", "Tissue", "hpi", "Run"] + all_proteins])
full_df = pd.concat(merged_dfs, axis=0).reset_index(drop=True)
full_df

In [ ]:
df_all_values = full_df.dropna(axis=1)
proteins_w_values = df_all_values.columns.difference(["nhpi", "Model", "Tissue", "hpi", "Run"])
df_all_values

In [ ]:
batch_means = df_all_values.groupby(["Tissue", "Model"])[proteins_w_values].transform("mean")
mean_of_means = batch_means.mean()
df_all_values[proteins_w_values] = df_all_values[proteins_w_values] - batch_means + mean_of_means

embedder = umap.UMAP(n_components=2, random_state=42, min_dist=.5, n_neighbors=15)
embedding = embedder.fit_transform(df_all_values[proteins_w_values])
embedding_df = pd.DataFrame(embedding, columns=["UMAP1", "UMAP2"])
embedding_df = pd.concat([df_all_values[["Model", "Tissue", "nhpi", "hpi"]].reset_index(drop=True), embedding_df], axis=1)
fig, axs = plt.subplots(1, 3, figsize=(8, 3), sharex=True, sharey=True)
sns.scatterplot(data=embedding_df, x="UMAP1", y="UMAP2", hue="Tissue", ax=axs[0])
sns.scatterplot(data=embedding_df, x="UMAP1", y="UMAP2", hue="nhpi", ax=axs[1])
sns.scatterplot(data=embedding_df, x="UMAP1", y="UMAP2", hue="Model", ax=axs[2])
for ax in axs:
    sns.despine(ax=ax)
    ax.legend(frameon=False)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()


In [ ]:
plt.figure(figsize=(3,3))
sns.histplot(data=full_df, x="nhpi", fill=True,bins=10, kde=True, common_norm=False)
plt.legend(frameon=False)
sns.despine()
plt.xlabel("Severity")
